# Run the agent on Kaggle

Runs the fine-tuning agent once, unattended, with the default settings (10 experiments, at most 40 LLM steps).
It works under **Save & Run All**.

**Before running:**
1. *Settings → Accelerator*: pick a GPU (one GPU is used even on 2×T4). *Settings → Internet*: on.
2. *Add Input*: attach the dataset, a folder with `images/` and `train_labels.csv`, and set `DATASET_PATH` below.
3. *Add-ons → Secrets*: add `API_KEY`. `BASE_URL` and `LLM_MODEL` are optional (OpenAI and `gpt-4.1-mini` by default).

Outputs go to `/kaggle/working/runs`: one checkpoint folder per experiment, plus `history.txt`.

In [ ]:
REPO_URL = "https://github.com/sinatb/breast-cancer-agent.git"
BRANCH = "overhaul"
DATASET_PATH = "/kaggle/input/<dataset>"  # edit: the folder that holds images/ and train_labels.csv
MODEL_DIR = "/kaggle/working/runs"
CODE_DIR = "/tmp/breast-cancer-agent"  # outside /kaggle/working, so the code is not saved as output

In [ ]:
!rm -rf {CODE_DIR}
!git clone -q --depth 1 -b {BRANCH} {REPO_URL} {CODE_DIR}
!git -C {CODE_DIR} rev-parse HEAD

In [ ]:
# Project dependencies only (no torch or torchvision); Kaggle's numpy already meets the numpy>=1.26 floor.
!pip install -q uv
!uv pip install --system -q -r {CODE_DIR}/pyproject.toml

In [ ]:
import sys

sys.path.insert(0, CODE_DIR)

import cv2
import peft
import timm
import torch

print("PyTorch:", torch.__version__, "| OpenCV:", cv2.__version__, "| timm:", timm.__version__, "| peft:", peft.__version__)
assert torch.cuda.is_available(), "No GPU: turn on a GPU accelerator in the notebook settings."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

from config import load_settings

secrets = UserSecretsClient()
for name in ("API_KEY", "BASE_URL", "LLM_MODEL"):
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        if name == "API_KEY":
            raise RuntimeError("Add the API_KEY secret (Add-ons -> Secrets) and attach it to this notebook.") from None

os.environ["DATASET_PATH"] = DATASET_PATH
os.environ["MODEL_DIR"] = MODEL_DIR
settings = load_settings()
print("Dataset:", settings.dataset_path, "| Runs:", settings.model_dir, "| LLM:", settings.llm_model)

In [ ]:
from agent.Orchestrator import Orchestrator
from main import run_session

orch = Orchestrator(settings)
result = run_session(orch, settings.model_dir / "history.txt")  # writes history.txt even if the run crashes
print(f"Session complete: {result.stop_reason} ({result.experiments} experiments, {result.steps} steps).")

In [ ]:
print(orch.memory.get_summary())
for run_dir in sorted(p for p in settings.model_dir.iterdir() if p.is_dir()):
    print(run_dir.name)